### STATIONARY TEST
| Test Name | Null Hypothesis | Interpretation |
|-----------|----------------|----------------|
| ADF | Non-stationary (unit root) | p < 0.05: stationary |
| KPSS | Stationary | p > 0.1: stationary |

In [ ]:
library(tseries)   # adf.test, kpss.test
library(ggplot2)
options(warn=-1)

In [ ]:
# Helper: format numbers to 3 d.p.
sci_fmt <- function(x) {
    tryCatch(sprintf("%.3f", as.numeric(x)), error = function(e) as.character(x))
}

## Helper Function

In [ ]:
stationarity_test <- function(ts_vec, lags = NULL) {
    results <- list()

    # ADF Test
    adf_res <- if (!is.null(lags)) adf.test(ts_vec, k=lags) else adf.test(ts_vec)
    adf_stat <- as.character(round(adf_res$statistic, 4))
    adf_p    <- as.character(round(adf_res$p.value, 4))
    adf_stat_text <- if (adf_res$p.value <= 0.05) "Yes" else "No"
    adf_msg <- if (adf_res$p.value <= 0.05)
        "Reject Null. No unit root and is stationary"
    else
        "Accept Null. Unit root, indicating it is non-stationary"
    cat("ADF:", adf_msg, "\n")
    results[["ADF"]] <- data.frame(
        "Test Statistic" = adf_stat, "P-Value" = adf_p,
        "Null.Hypo" = "Non-stationary (Has unit root)",
        "Stationary" = adf_stat_text, "Details" = adf_msg,
        check.names=FALSE)

    # KPSS Test
    kpss_res <- if (!is.null(lags)) kpss.test(ts_vec, lshort=FALSE) else kpss.test(ts_vec)
    kpss_stat <- as.character(round(kpss_res$statistic, 4))
    kpss_p    <- as.character(round(kpss_res$p.value, 4))
    kpss_stat_text <- if (kpss_res$p.value > 0.1) "Yes" else "No"
    kpss_msg <- if (kpss_res$p.value > 0.1)
        "Fail to reject Null. No unit root and is stationary"
    else
        "Reject Null. Has unit root, indicating it is non-stationary"
    cat("KPSS:", kpss_msg, "\n")
    results[["KPSS"]] <- data.frame(
        "Test Statistic" = kpss_stat, "P-Value" = kpss_p,
        "Null.Hypo" = "Stationary (No unit root)",
        "Stationary" = kpss_stat_text, "Details" = kpss_msg,
        check.names=FALSE)

    do.call(rbind, results)
}

## Time Series in levels

In [ ]:
usd_cad <- read.csv("../../../data/USD_CAD.csv")
rownames(usd_cad) <- usd_cad$time

plot(usd_cad$m_c, type="l", main="USD/CAD Level", xlab="", ylab="Rate")
grid()

In [ ]:
stationarity_test(usd_cad$m_c)

## Time Series in log difference (log returns)

In [ ]:
ts_diff <- diff(log(usd_cad$m_c))
plot(ts_diff, type="l", main="USD/CAD Log Difference", xlab="", ylab="Log Return")
grid()

In [ ]:
stationarity_test(ts_diff, lags=6)